In [3]:
import json
import os
from collections import Counter

from datasets import Dataset
from transformers import AutoTokenizer

In [8]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

DATA_PATH = "datset\\model2_dummy.json"

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"

OUTPUT_DIR = "model2_preprocessed"

MAX_LENGTH = 512

In [9]:
# ============================================================
# 2. LOAD DATASET
# ============================================================

print("Loading dataset...")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} examples.")

Loading dataset...
Loaded 100 examples.


In [10]:
# ============================================================
# 3. VALIDATE DATASET
# ============================================================

print("Validating dataset...")

for i, example in enumerate(data):

    if "context" not in example:
        raise ValueError(f"Example {i} is missing 'context'.")

    if "target" not in example:
        raise ValueError(f"Example {i} is missing 'target'.")

    if "safety" not in example:
        raise ValueError(f"Example {i} is missing 'safety'.")

    if not isinstance(example["context"], list):
        raise ValueError(f"Example {i}: 'context' must be a list.")

    if "red_flag" not in example["safety"]:
        raise ValueError(f"Example {i}: missing 'safety.red_flag'.")

print("Dataset validation passed.")

Validating dataset...
Dataset validation passed.


In [12]:
# ============================================================
# 4. CONVERT EACH EXAMPLE INTO:
#
#    conversation text
#    label
# ============================================================

processed_data = []

for example in data:

    # --------------------------------------------------------
    # Build conversation text
    # --------------------------------------------------------

    conversation_parts = []

    for message in example["context"]:

        role = message["role"].capitalize()
        content = message["content"]

        conversation_parts.append(
            f"{role}: {content}"
        )

    conversation_text = "\n".join(conversation_parts)


    # --------------------------------------------------------
    # Determine label
    # --------------------------------------------------------

    if example["safety"]["red_flag"] is True:

        # Special label for emergency/red-flag cases
        label_name = "RED_FLAG"

    else:

        label_name = example["target"]

        if label_name is None:
            raise ValueError(
                f"Example {example['id']} has target=None "
                f"but is not marked as a red flag."
            )


    processed_data.append(
        {
            "id": example["id"],
            "text": conversation_text,
            "label_name": label_name
        }
    )

In [13]:
# ============================================================
# 5. CREATE LABEL MAPPING
# ============================================================

print("Creating label mapping...")

all_labels = sorted(
    set(example["label_name"] for example in processed_data)
)

label2id = {
    label: idx
    for idx, label in enumerate(all_labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}


print("\nLabels found:")

for label, idx in label2id.items():
    print(f"{idx}: {label}")

Creating label mapping...

Labels found:
0: RED_FLAG
1: associated_symptoms
2: breath_onset
3: breath_progression
4: breath_severity
5: cough_character
6: cough_frequency
7: cough_progression
8: diarrhea_frequency
9: dizziness_frequency
10: dizziness_onset
11: dizziness_progression
12: dizziness_triggers
13: dizziness_type
14: duration
15: fever_progression
16: fever_temperature
17: functional_impact
18: injury_or_trigger
19: nausea_onset
20: pain_location
21: pain_progression
22: pain_quality
23: pain_relieving_factors
24: pain_severity
25: pain_triggers
26: rash_character
27: rash_location
28: rash_progression
29: severity
30: throat_severity
31: urinary_frequency
32: urinary_onset
33: urinary_progression
34: vomiting
35: vomiting_frequency


In [14]:
# ============================================================
# 6. CONVERT LABEL NAMES → INTEGER LABELS
# ============================================================

for example in processed_data:

    example["labels"] = label2id[
        example["label_name"]
    ]


In [15]:
# ============================================================
# 7. SHOW LABEL DISTRIBUTION
# ============================================================

label_counts = Counter(
    example["label_name"]
    for example in processed_data
)

print("\nLabel distribution:")

for label, count in label_counts.items():
    print(f"{label}: {count}")


Label distribution:
pain_location: 7
pain_quality: 5
pain_severity: 6
associated_symptoms: 25
pain_progression: 2
pain_triggers: 1
pain_relieving_factors: 1
functional_impact: 8
cough_character: 2
cough_frequency: 1
cough_progression: 1
dizziness_onset: 1
dizziness_frequency: 1
dizziness_progression: 1
dizziness_triggers: 1
dizziness_type: 1
throat_severity: 1
fever_temperature: 1
fever_progression: 1
injury_or_trigger: 3
nausea_onset: 1
vomiting: 1
vomiting_frequency: 1
diarrhea_frequency: 1
severity: 1
duration: 1
breath_onset: 1
breath_severity: 1
breath_progression: 1
urinary_onset: 1
urinary_frequency: 1
urinary_progression: 1
rash_location: 1
rash_character: 1
rash_progression: 1
RED_FLAG: 15


In [16]:
# ============================================================
# 8. CREATE HUGGING FACE DATASET
# ============================================================

dataset = Dataset.from_list(processed_data)

print("\nDataset created:")
print(dataset)


Dataset created:
Dataset({
    features: ['id', 'text', 'label_name', 'labels'],
    num_rows: 100
})


In [17]:
# ============================================================
# 9. LOAD CLINICALBERT TOKENIZER
# ============================================================

print("\nLoading ClinicalBERT tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")


Loading ClinicalBERT tokenizer...


c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\astha\.cache\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer loaded.


In [18]:
# ============================================================
# 10. TOKENIZE
# ============================================================

print("\nTokenizing dataset...")


def tokenize_function(batch):

    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True
)


print("Tokenization complete.")


Tokenizing dataset...


Map: 100%|██████████| 100/100 [00:00<00:00, 6148.02 examples/s]

Tokenization complete.


In [21]:
# 11. CREATE OUTPUT DIRECTORY

OUTPUT_DIR = "model2_preprocessed"

In [22]:
# ============================================================
# 12. SAVE TOKENIZED DATASET
# ============================================================

dataset_output_path = os.path.join(
    OUTPUT_DIR,
    "dataset"
)

tokenized_dataset.save_to_disk(
    dataset_output_path
)

print(
    f"\nProcessed dataset saved to: "
    f"{dataset_output_path}"
)

Saving the dataset (1/1 shards): 100%|██████████| 100/100 [00:00<00:00, 9527.53 examples/s] 


Processed dataset saved to: model2_preprocessed\dataset


In [23]:
# ============================================================
# 13. SAVE TOKENIZER
# ============================================================

tokenizer_output_path = os.path.join(
    OUTPUT_DIR,
    "tokenizer"
)

tokenizer.save_pretrained(
    tokenizer_output_path
)

print(
    f"Tokenizer saved to: "
    f"{tokenizer_output_path}"
)


Tokenizer saved to: model2_preprocessed\tokenizer


In [24]:
# ============================================================
# 14. SAVE LABEL MAPPING
# ============================================================

label_mapping_path = os.path.join(
    OUTPUT_DIR,
    "label_mapping.json"
)

with open(
    label_mapping_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "label2id": label2id,
            "id2label": {
                str(k): v
                for k, v in id2label.items()
            }
        },
        f,
        indent=4
    )


print(
    f"Label mapping saved to: "
    f"{label_mapping_path}"
)


Label mapping saved to: model2_preprocessed\label_mapping.json


In [25]:
# ============================================================
# 15. DISPLAY ONE PROCESSED EXAMPLE
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE PROCESSED EXAMPLE")
print("=" * 60)

sample = tokenized_dataset[0]

print("\nOriginal text:")
print(sample["text"])

print("\nLabel name:")
print(sample["label_name"])

print("\nLabel ID:")
print(sample["labels"])

print("\nInput IDs:")
print(sample["input_ids"][:20])

print("\nAttention mask:")
print(sample["attention_mask"][:20])

print("\nPreprocessing finished successfully!")


SAMPLE PROCESSED EXAMPLE

Original text:
User: I've had stomach pain since yesterday.

Label name:
pain_location

Label ID:
20

Input IDs:
[101, 4795, 131, 178, 112, 1396, 1125, 3472, 2489, 1290, 8128, 119, 102]

Attention mask:
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Preprocessing finished successfully!
